In [ ]:
%pip install -q langchain langchain-aws langgraph boto3 python-dotenv

# Week 6 Tuesday -- Context Engineering

## Overview

On Monday we built agents with `create_agent()` and orchestrated multi-step workflows with `StateGraph`. We controlled **the order** in which nodes execute. Today we shift focus to **what each node sees** -- the information the model receives, the tools it can access, and the data those tools can read and write.

This discipline is called **context engineering**, and it is the single biggest lever you have for making agents behave reliably in production.

## Learning Objectives

By the end of today you will be able to:

- Explain what context engineering is and why agent failures usually trace back to context problems
- Use `@dynamic_prompt` to build system prompts that adapt at runtime
- Use `@wrap_model_call` to inject messages, filter tools, and select models dynamically
- Use `ToolRuntime` to give tools access to state, store, and config
- Return `Command` from a tool to make persistent state updates
- Combine multiple middleware into a layered stack for production agents

## Stages

| Stage | Topic | Demo | Readings |
|-------|-------|------|----------|
| 1 | Context Engineering Introduction & Dynamic Prompts | `demo_01` | 01 - Context Engineering Introduction |
| 2 | Wrapping Model Calls & Model Context | `demo_02` | 02 - Model Context (Transient Changes) |
| 3 | Tool Context & ToolRuntime | `demo_03` | 03 - Tool Context (Persistent Changes), 04 - Middleware Patterns |

---

## Stage 1 -- Context Engineering Introduction & Dynamic Prompts

Context engineering is the art and science of **controlling what information an agent sees and when**. You have built agents that work -- sometimes. But you have also seen agents that forget what the user said five messages ago, use the wrong tool, hallucinate facts, or ignore the system prompt. Every one of those failures traces back to a context problem.

### The Agent Loop

To understand where context engineering applies, trace the agent loop:

```
┌─────────────────────────────────────────────────────────────┐
│                     THE AGENT LOOP                          │
│                                                             │
│  1. RECEIVE     User sends a message or system triggers     │
│       │                                                     │
│       ▼                                                     │
│  2. PREPARE     Build context: system prompt, history,      │
│       │         current message, available tools            │
│       ▼                                                     │
│  3. CALL MODEL  Send context to LLM, get response           │
│       │                                                     │
│       ▼                                                     │
│  4. PROCESS     Parse response: text answer or tool calls   │
│       │                                                     │
│       ├──── If tool call ────┐                              │
│       ▼                      ▼                              │
│  5a. RETURN     5b. EXECUTE TOOLS                           │
│     answer         │                                        │
│                    ▼                                        │
│               Add tool results to context                   │
│                    │                                        │
│                    └──────► Back to step 3                  │
└─────────────────────────────────────────────────────────────┘
```

Context engineering happens at **steps 2, 4, and 5b** -- preparing what the model sees, processing what it returns, and handling tool execution.

### The Three Types of Context

LangChain organizes context into three categories:

1. **Model Context (Transient)** -- What the model sees during a single request: system prompt, conversation history, available tools, injected context. Changes apply to one request but do not persist in state. Middleware: `@dynamic_prompt`, `@wrap_model_call`.

2. **Tool Context (Persistent)** -- How tools read and write data. Tools access state, store, and config via `ToolRuntime`. They return `Command` to update state persistently. Changes carry across turns.

3. **Life-cycle Context (Cross-cutting)** -- Hooks that run before/after key operations: `@before_model`, `@after_model`, `@before_tools`, `@after_tools`.

### Why Agents Fail

Most agent failures map directly to context problems:

| Failure | Context Problem | Solution |
|---------|-----------------|----------|
| Forgets earlier conversation | History too short or summarized poorly | `SummarizationMiddleware`, proper truncation |
| Uses wrong tool | Tool descriptions unclear | Better docstrings, `@wrap_model_call` to filter |
| Doesn't follow instructions | System prompt overwritten or too long | `@dynamic_prompt` for focused prompts |
| Misses user preferences | Context not injected | `ToolRuntime` to read from store |
| Hallucinates facts | No retrieval context | Pre-retrieval tools, injected context |

Think of context engineering as **designing a conversation** rather than just sending prompts. Without it you hand the LLM a kitchen sink of information and hope for the best. With it you curate exactly what the model sees so its behavior is focused and predictable.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")
print("Model initialized:", model)

### The `@dynamic_prompt` Decorator

The simplest model context middleware is `@dynamic_prompt`. It replaces a static system prompt with a function that runs **before every model call**. The function receives a `ModelRequest` object and returns the prompt string.

`ModelRequest` exposes:
- `request.messages` -- all messages in the conversation
- `request.state` -- current workflow state
- `request.config` -- configuration passed at invocation
- `request.tools` -- currently available tools

This lets you tailor the system prompt based on conversation length, user tier, time of day, or anything else you can read at runtime.

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    weather_data = {
        "new york": "72F, Sunny",
        "london": "58F, Cloudy",
        "tokyo": "68F, Clear"
    }
    return weather_data.get(city.lower(), f"Weather data for {city}: 65F, Partly Cloudy")


@tool
def get_time(timezone: str) -> str:
    """Get current time in a timezone."""
    from datetime import datetime
    return f"Current time in {timezone}: {datetime.now().strftime('%I:%M %p')}"


print("Tools defined:", get_weather.name, get_time.name)

Below we define two `@dynamic_prompt` functions. The first adapts the system prompt based on conversation length -- welcoming at first, then progressively more concise as the conversation grows. The second adapts based on time of day.

Only **one** `@dynamic_prompt` should be active at a time since each one replaces the system prompt entirely.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dynamic_prompt
def adaptive_conversation_prompt(request: ModelRequest) -> str:
    message_count = len(request.messages)

    base = "You are a helpful assistant with access to weather and time tools."

    if message_count < 5:
        return base + " Welcome the user warmly. Let them know you can provide weather and time information."
    elif message_count < 10:
        return base + " Provide detailed, helpful responses with a positive attitude."
    elif message_count < 15:
        return base + " Be helpful but concise."
    else:
        return base + " Be extremely concise."


print("adaptive_conversation_prompt middleware ready")

In [ ]:
@dynamic_prompt
def time_aware_prompt(request: ModelRequest) -> str:
    from datetime import datetime

    hour = datetime.now().hour

    if 5 <= hour < 12:
        greeting = "Good morning!"
    elif 12 <= hour < 17:
        greeting = "Good afternoon!"
    elif 17 <= hour < 21:
        greeting = "Good evening!"
    else:
        greeting = "Hello, night owl!"

    return f"You are a friendly assistant. {greeting} Help users with weather and time."


print("time_aware_prompt middleware ready")

Now we create an agent that uses the adaptive prompt middleware. Notice that `create_agent` accepts a `middleware` list. We wire in `adaptive_conversation_prompt`; swap in `time_aware_prompt` to see the alternative behavior.

We also attach an `InMemorySaver` checkpointer so the agent remembers previous turns within the same thread.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


agent_01 = create_agent(
    name="dynamic_prompt_demo",
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[get_weather, get_time],
    middleware=[
        adaptive_conversation_prompt,
        # time_aware_prompt,  # swap in to try the alternative
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "demo-01"}}

messages = [
    "Hi there!",
    "What's the weather in Tokyo?",
    "And in New York?",
]

for msg in messages:
    result = agent_01.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config
    )
    print(f"User: {msg}")
    print(f"Assistant: {result['messages'][-1].content}\n")

---

## Stage 2 -- Wrapping Model Calls & Model Context

`@dynamic_prompt` controls the system prompt. When you need deeper control -- injecting messages, filtering tools, or selecting a different model -- reach for `@wrap_model_call`.

### How `@wrap_model_call` Works

A `@wrap_model_call` function receives the full `ModelRequest` plus a `handler` callback. You modify the request however you like, then call `handler(modified_request)` to continue. This gives you the ability to:

- **Inject system messages** with user profile data or retrieved context
- **Filter the tool list** so the model only sees tools the user is authorized to use
- **Swap the model** based on task complexity or user tier
- **Modify messages** to add, remove, or transform conversation history

```
User Request
     │
     ▼
┌─────────────────────┐
│  @dynamic_prompt    │──▶ Modify system prompt
└─────────────────────┘
     │
     ▼
┌─────────────────────┐
│  @wrap_model_call   │──▶ Modify messages, tools, model
└─────────────────────┘
     │
     ▼
   LLM Call
```

Middleware runs in the order specified in the `middleware` list. Each can modify the request before passing it to the next.

### State-Driven Tool Access

In this demo we define a custom `State` with `user_tier` and `user_name` fields. We also define three tools at different access levels:

- `search_public_docs` -- available to everyone
- `search_internal_docs` -- premium users and above
- `execute_query` -- admin only

Two `@wrap_model_call` middleware functions will (1) inject user context as a system message and (2) filter the tool list based on the user's tier. The model literally cannot call a tool it is not authorized to use because it never sees it.

In [ ]:
from typing import Callable, Literal, Annotated
from typing_extensions import TypedDict
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_tier: Literal["free", "premium", "admin"]
    user_name: str


@tool
def search_public_docs(query: str) -> str:
    """Search public documentation. Available to all users."""
    return f"Public docs result for '{query}': Found 3 relevant articles."


@tool
def search_internal_docs(query: str) -> str:
    """Search internal company docs. PREMIUM ONLY."""
    return f"Internal docs result for '{query}': Found confidential project details."


@tool
def execute_query(sql: str) -> str:
    """Execute a database query. ADMIN ONLY."""
    return f"Query executed: {sql}\nReturned 42 rows."


FREE_TOOLS = [search_public_docs]
PREMIUM_TOOLS = [search_public_docs, search_internal_docs]
ADMIN_TOOLS = [search_public_docs, search_internal_docs, execute_query]

print("State schema and tiered tools defined")

### Context Injection Middleware

The first middleware below injects a system message containing the user's name and tier. The model sees this context every turn without us having to stuff it into the system prompt.

### Permission-Based Tool Filtering

The second middleware reads the user's tier from state and overrides the tool list. Free users only see `search_public_docs`; premium users also get `search_internal_docs`; admins get everything including `execute_query`.

In [ ]:
@wrap_model_call
async def inject_user_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    user_tier = request.state.get("user_tier", "free")
    user_name = request.state.get("user_name", "Guest")

    context_message = {
        "role": "system",
        "content": f"Current user: {user_name} (Tier: {user_tier})"
    }

    messages = [context_message] + list(request.messages)
    modified_request = request.override(messages=messages)

    return await handler(modified_request)


print("inject_user_context middleware ready")

In [ ]:
@wrap_model_call
async def permission_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    user_tier = request.state.get("user_tier", "free")

    if user_tier == "admin":
        tools = ADMIN_TOOLS
    elif user_tier == "premium":
        tools = PREMIUM_TOOLS
    else:
        tools = FREE_TOOLS

    modified_request = request.override(tools=tools)

    return await handler(modified_request)


print("permission_based_tools middleware ready")

### Testing with Different User Tiers

We create a single agent with both middleware functions and then invoke it three times -- once as a free user (Alice), once as premium (Bob), and once as admin (Charlie). Watch how the available tools and injected context change the model's behavior for each tier.

In [ ]:
agent_02 = create_agent(
    name="wrap_model_call_demo",
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=ADMIN_TOOLS,
    system_prompt=(
        "You are a helpful research assistant. "
        "Use available tools to help users find information. "
        "If a user asks for something you can't access, explain why."
    ),
    middleware=[
        inject_user_context,
        permission_based_tools,
    ],
    state_schema=State,
)

test_cases = [
    {"user_tier": "free", "user_name": "Alice"},
    {"user_tier": "premium", "user_name": "Bob"},
    {"user_tier": "admin", "user_name": "Charlie"},
]

query = "Search for project details and run a database query"

for state in test_cases:
    print(f"\n=== Testing as {state['user_name']} ({state['user_tier']}) ===")
    result = agent_02.invoke({
        "messages": [{"role": "user", "content": query}],
        **state
    })
    print(f"Response: {result['messages'][-1].content[:300]}...")

---

## Stage 3 -- Tool Context & ToolRuntime

Model context middleware controls what the LLM *sees* during inference. Tool context controls what tools can *do* during execution. Tools often need to:

- **Read** user preferences, permissions, or cached data from state
- **Write** updates that persist beyond the current turn
- **Access** long-term memory that spans conversations

`ToolRuntime` and `Command` solve these problems without polluting the LLM's attention window.

### ToolRuntime

When you add a `runtime: ToolRuntime` parameter to any `@tool` function, LangChain automatically injects it at execution time. Through `runtime` you can access:

```
┌────────────────────────────────────────────────────────────┐
│                     ToolRuntime                             │
│                                                            │
│  ┌───────────────┐  ┌───────────────┐  ┌───────────────┐  │
│  │    state      │  │    store      │  │    config     │  │
│  │               │  │               │  │               │  │
│  │ Current       │  │ Long-term     │  │ Invocation    │  │
│  │ workflow      │  │ memory        │  │ settings      │  │
│  │ state         │  │ (cross-conv)  │  │               │  │
│  └───────────────┘  └───────────────┘  └───────────────┘  │
└────────────────────────────────────────────────────────────┘
```

- `runtime.state` -- the current workflow state (your `TypedDict`)
- `runtime.store` -- long-term memory that persists across conversations
- `runtime.config` -- configuration passed at invocation time
- `runtime.tool_call_id` -- the ID of the current tool call (needed when returning `Command`)

### Command: Persistent State Updates

Normally a tool returns a string that goes back to the model as a `ToolMessage`. When a tool returns a `Command` instead, it can update state fields persistently. This is the bridge between transient model context and persistent tool context.

| Transient (Model Context) | Persistent (Tool Context) |
|---------------------------|---------------------------|
| `@dynamic_prompt` | `Command(update={...})` |
| `@wrap_model_call` | `runtime.store.put()` |
| Affects single LLM call | Affects future turns |
| No permanent record | Persists across turns/sessions |

Use **transient** for prompt customization, tool filtering, model selection. Use **persistent** for search history, user preferences, accumulated results, session state.

### Building Tools with ToolRuntime

We will build four tools that demonstrate reading and writing context:

1. `check_session_info` -- reads from `runtime.state` to report session data
2. `update_preferences` -- returns a `Command` to persistently update state
3. `save_note` -- writes to `runtime.store` for cross-conversation memory
4. `get_note` -- reads from `runtime.store` to retrieve saved notes

In [ ]:
from langchain.tools import ToolRuntime
from langgraph.types import Command
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import ToolMessage
from datetime import datetime


class SessionState(TypedDict):
    user_id: str
    session_started: bool
    messages: Annotated[list[AnyMessage], add_messages]


@tool
def check_session_info(runtime: ToolRuntime) -> str:
    """Check current session information."""
    session_started = runtime.state.get("session_started", False)
    message_count = len(runtime.state.get("messages", []))
    user_id = runtime.state.get("user_id", "anonymous")

    return (
        f"Session Info:\n"
        f"- User ID: {user_id}\n"
        f"- Session started: {session_started}\n"
        f"- Messages in conversation: {message_count}"
    )


print("check_session_info tool defined")

In [ ]:
@tool
def update_preferences(
    preference_key: str,
    preference_value: str,
    runtime: ToolRuntime[SessionState]
) -> Command:
    """Update a user preference in the session."""
    return Command(
        update={
            "messages": [ToolMessage("Success", tool_call_id=runtime.tool_call_id)],
            preference_key: preference_value
        }
    )


print("update_preferences tool defined")

In [ ]:
@tool
def save_note(
    note_title: str,
    note_content: str,
    runtime: ToolRuntime
) -> str:
    """Save a note to long-term memory (store). Notes persist across conversations."""
    user_id = runtime.state.get("user_id") or "default"

    runtime.store.put(
        ("users", user_id, "notes"),
        note_title,
        {
            "content": note_content,
            "created_at": datetime.now().isoformat()
        }
    )

    return f"Saved note '{note_title}' to long-term memory."


@tool
def get_note(
    note_title: str,
    runtime: ToolRuntime
) -> str:
    """Retrieve a specific note by title from long-term memory."""
    user_id = runtime.state.get("user_id") or "default"

    result = runtime.store.get(
        ("users", user_id, "notes"),
        note_title
    )

    if result:
        return f"Note '{note_title}': {result.value.get('content', 'No content')}"
    return f"No note found with title '{note_title}'"


print("save_note and get_note tools defined")

### Testing ToolRuntime and Command

We create an agent wired to all four tools, backed by an `InMemorySaver` (for state persistence across turns) and an `InMemoryStore` (for long-term cross-conversation memory). We then exercise the tools: check session info, save a note, and retrieve it.

In [ ]:
store = InMemoryStore()

agent_03 = create_agent(
    name="tool_runtime_demo",
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[check_session_info, update_preferences, save_note, get_note],
    system_prompt=(
        "You are a personal assistant with memory capabilities. "
        "You can check session info, update user preferences, "
        "save notes to long-term memory, and retrieve notes by title. "
        "Always confirm when you have saved information."
    ),
    checkpointer=InMemorySaver(),
    store=store,
    state_schema=SessionState,
)

config = {"configurable": {"thread_id": "demo_thread", "user_id": "demo_user"}}

result = agent_03.invoke(
    {"messages": [{"role": "user", "content": "Check my session info"}]},
    config
)
print("Session check:", result["messages"][-1].content)

result = agent_03.invoke(
    {"messages": [{"role": "user", "content": "Save a note titled 'Meeting' with content 'Discuss Q4 goals'"}]},
    config
)
print("\nSave note:", result["messages"][-1].content)

result = agent_03.invoke(
    {"messages": [{"role": "user", "content": "Retrieve my note titled 'Meeting'"}]},
    config
)
print("\nGet note:", result["messages"][-1].content)

---

## Middleware Patterns for Production

With all three types of context in hand, here are the key production patterns to internalize:

### Pattern 1: Permission-Based Tool Selection

Define a mapping from tool names to required roles and filter dynamically inside `@wrap_model_call`. Users never see tools they cannot use.

```python
TOOL_PERMISSIONS = {
    "search_public": [],
    "search_internal": ["employee"],
    "modify_record": ["editor", "admin"],
    "delete_record": ["admin"],
}

@wrap_model_call
def permission_filter(request: ModelRequest, handler):
    roles = request.state.get("roles", [])
    allowed = [
        t for t in request.tools
        if not TOOL_PERMISSIONS.get(t.name)
        or any(r in roles for r in TOOL_PERMISSIONS[t.name])
    ]
    return handler(request.override(tools=allowed))
```

### Pattern 2: Context Injection from External Sources

Fetch data from a database or API inside `@wrap_model_call` and inject it as a system message. The model gets user profile information, recent activity, or retrieved documents without your application code managing the conversation manually.

### Pattern 3: Dynamic Model Routing

Analyze the latest message for keywords or length and swap the model via `request.override(model=...)`. Use a cheaper model for simple queries and a stronger model for complex reasoning or code generation.

```python
@wrap_model_call
def smart_router(request: ModelRequest, handler):
    content = request.messages[-1].content if request.messages else ""
    is_complex = len(content) > 500 or "explain" in content.lower()
    model = "openai:gpt-4o" if is_complex else "openai:gpt-4o-mini"
    return handler(request.override(model=model))
```

### Pattern 4: Conversation-Aware Prompting

Detect frustration, repeated questions, or long conversations in `@dynamic_prompt` and adapt tone, detail level, or escalation behavior accordingly.

### Pattern 5: Layered Middleware Stack

Combine all of the above into a single `middleware` list. Order matters -- each middleware transforms the request before passing it to the next:

```python
agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[...],
    middleware=[
        conversation_aware_prompt,   # 1. Adapt system prompt
        inject_user_profile,         # 2. Add user context
        permission_based_tools,      # 3. Filter tools
        smart_model_router,          # 4. Select model
        SummarizationMiddleware(...) # 5. Handle long conversations
    ],
    name="production_agent"
)
```

---

## Key Takeaways

1. **Context engineering** is the discipline of controlling what information an agent sees and when. Most agent failures -- wrong tool, forgotten context, hallucinated facts -- trace back to context problems.

2. **`@dynamic_prompt`** replaces a static system prompt with a function that runs before every model call. Use it to adapt tone, instructions, or focus based on conversation length, user tier, or time of day.

3. **`@wrap_model_call`** gives full control over the model request. Inject system messages, filter the tool list by permissions, or swap models based on task complexity -- all without changing your core agent logic.

4. **`ToolRuntime`** gives tools access to `state`, `store`, and `config` at execution time. Tools can read user preferences, session data, or long-term memory without cluttering the LLM's context window.

5. **`Command`** lets tools make persistent state updates that carry across turns. This is the bridge between transient model context and durable application state.

6. **Middleware composes cleanly.** Stack `@dynamic_prompt`, multiple `@wrap_model_call` functions, and built-in middleware like `SummarizationMiddleware` in a single `middleware` list. Order matters -- each layer transforms the request before the next sees it.

## Up Next: Wednesday -- Multi-Agent Systems

Tomorrow we move from single agents with middleware to **coordinated teams of agents**. You will learn how to break complex tasks across specialized agents, route work between them, and apply today's context engineering patterns to ensure each agent sees exactly the context it needs. Everything you learned today -- dynamic prompts, tool filtering, persistent state updates -- becomes even more powerful when agents hand off to one another.